# R2Gen-KG: Training & Testing on IU X-Ray

This notebook trains the R2Gen model with Knowledge Graph (KG) and Contrastive Attention (CA) on the IU X-Ray dataset, then evaluates on the test split.

**Pipeline:**
1. Setup environment & args
2. Load tokenizer and dataloaders
3. Build model (auto-constructs KG during init)
4. Build contrastive attention normality pool
5. Stage 1 — KG pretraining (optional)
6. Stage 2 — Full report generation training
7. Evaluation on test set

## 1. Imports & Environment

In [ ]:
import os, sys
from pathlib import Path

# ── Colab setup ──────────────────────────────────────────────────────────────
# If the repo isn't cloned yet, uncomment ONE of the options below:

# Option A — clone from GitHub (replace with your repo URL)
# !git clone https://github.com/YOUR_USERNAME/R2Gen-main.git /content/R2Gen-main

# Option B — mount Google Drive and point to it
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_ROOT = Path("/content/drive/MyDrive/R2Gen-main")  # adjust as needed

# Option C — repo already cloned at /content/R2Gen-main
REPO_ROOT = Path("/content/R2Gen-main")
# ─────────────────────────────────────────────────────────────────────────────

import json, random, argparse
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

assert (REPO_ROOT / "data" / "iu_xray" / "annotation.json").exists(), \
    f"annotation.json not found under {REPO_ROOT} — check your REPO_ROOT."

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root : {REPO_ROOT}")
print(f"CWD       : {os.getcwd()}")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 9233
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── hyperparameters ───────────────────────────────────────────────────────────
# Use argparse.Namespace so existing modules accept the config unchanged.

args = argparse.Namespace(
    # ── paths ──────────────────────────────────────────────────────────────
    image_dir       = "data/iu_xray/images/",
    ann_path        = "data/iu_xray/annotation.json",
    dataset_name    = "iu_xray",
    save_dir        = "results/iu_xray_kg",

    # ── tokeniser ──────────────────────────────────────────────────────────
    max_seq_length  = 60,
    threshold       = 3,       # min token frequency to include in vocab

    # ── data loading ───────────────────────────────────────────────────────
    batch_size      = 16,
    num_workers     = 0,       # set >0 if your OS supports shared memory

    # ── visual extractor ───────────────────────────────────────────────────
    # Options: 'resnet101'  →  d_vf=2048
    #          'medsam'     →  d_vf=256
    visual_extractor           = "resnet101",
    visual_extractor_pretrained = True,
    d_vf                       = 2048,

    # ── transformer ────────────────────────────────────────────────────────
    d_model         = 512,
    d_ff            = 512,
    num_heads       = 8,
    num_layers      = 3,
    dropout         = 0.1,
    drop_prob_lm    = 0.5,

    # ── decoding ───────────────────────────────────────────────────────────
    sample_method   = "beam_search",
    beam_size       = 3,
    temperature     = 1.0,
    sample_n        = 1,
    group_size      = 1,
    output_logsoftmax = 1,
    decoding_constraint = 0,
    block_trigrams  = 1,

    # ── knowledge graph ────────────────────────────────────────────────────
    kg_max_nodes           = 40,
    kg_min_term_freq       = 5,
    kg_co_occur_threshold  = 3,
    kg_pretrain_epochs     = 10,
    kg_loss_weight         = 0.1,
    biomedclip_device      = "cpu",   # change to 'cuda' if VRAM allows

    # ── contrastive attention ──────────────────────────────────────────────
    use_contrastive_attention = True,
    ca_pool_size              = 100,
    ca_num_rounds             = 3,
    ca_dropout                = 0.1,

    # ── optimiser ──────────────────────────────────────────────────────────
    optim           = "Adam",
    lr_ve           = 5e-5,    # visual extractor learning rate
    lr_ed           = 1e-4,    # encoder-decoder learning rate
    weight_decay    = 5e-5,
    amsgrad         = True,
    adam_betas      = (0.9, 0.98),
    adam_eps        = 1e-9,

    # ── LR scheduler ───────────────────────────────────────────────────────
    lr_scheduler    = "StepLR",
    step_size       = 50,
    gamma           = 0.1,

    # ── training ───────────────────────────────────────────────────────────
    epochs          = 100,
    save_period     = 1,
    monitor_metric  = "val_BLEU_4",
    monitor_mode    = "max",
    early_stop      = 50,
    seed            = SEED,
    resume          = None,    # path to checkpoint to resume from
)

# Auto-set d_vf if you switch visual_extractor to medsam
if args.visual_extractor == "medsam":
    args.d_vf = 256

os.makedirs(args.save_dir, exist_ok=True)
print("Config ready. save_dir:", args.save_dir)

## 3. Inspect Dataset

In [ ]:
with open(args.ann_path) as f:
    annotation = json.load(f)

for split in ("train", "val", "test"):
    print(f"{split:5s}: {len(annotation[split]):4d} studies")

# Preview one example
sample = annotation["train"][0]
print("\nSample keys:", list(sample.keys()))
print("Report:", sample["report"][:120], "...")
print("Images:", sample["image_path"])

In [ ]:
# Visual spot-check — show the two X-rays for a random study
idx = random.randint(0, len(annotation["train"]) - 1)
sample = annotation["train"][idx]
fig, axes = plt.subplots(1, len(sample["image_path"]), figsize=(10, 5))
if len(sample["image_path"]) == 1:
    axes = [axes]
for ax, img_rel in zip(axes, sample["image_path"]):
    img_path = os.path.join(args.image_dir, img_rel)
    ax.imshow(mpimg.imread(img_path), cmap="gray")
    ax.set_title(img_rel, fontsize=8)
    ax.axis("off")
fig.suptitle(f"Study {sample['id']}\n{sample['report'][:100]}...", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Tokenizer & Dataloaders

In [ ]:
from modules.tokenizers import Tokenizer
from modules.dataloaders import R2DataLoader

tokenizer = Tokenizer(args)
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")

In [ ]:
train_dataloader = R2DataLoader(args, tokenizer, split="train", shuffle=True)
val_dataloader   = R2DataLoader(args, tokenizer, split="val",   shuffle=False)
test_dataloader  = R2DataLoader(args, tokenizer, split="test",  shuffle=False)

print(f"Train batches : {len(train_dataloader)}")
print(f"Val batches   : {len(val_dataloader)}")
print(f"Test batches  : {len(test_dataloader)}")

# Sanity-check one batch shape
ids, images, reports_ids, reports_masks = next(iter(train_dataloader))
print(f"\nImages shape  : {images.shape}   # [B, 2, 3, H, W]")
print(f"Reports shape : {reports_ids.shape}  # [B, L]")

## 5. Build Model

In [ ]:
from models.r2gen_kg import R2GenKGModel

print("Building model (KG graph construction may take ~1–2 min on first run)...")
model = R2GenKGModel(args, tokenizer)
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params    : {total_params/1e6:.1f} M")
print(f"Trainable params: {trainable_params/1e6:.1f} M")

## 6. Build Contrastive Attention Normality Pool

In [ ]:
if args.use_contrastive_attention:
    ca_module = model.encoder_decoder.contrastive_attn
    if ca_module is not None and not ca_module.pool_built:
        print("Building normality pool (BiomedCLIP pass over train set)...")
        kg_builder = model.encoder_decoder.kg_builder
        ca_module.build_normality_pool(
            visual_extractor = model.visual_extractor,
            dataloader       = train_dataloader,
            dataset_name     = args.dataset_name,
            ann_path         = args.ann_path,
            device           = device,
            max_pool_size    = args.ca_pool_size,
            kg_builder       = kg_builder,
        )
        print(f"Normality pool built. Shape: {ca_module.normality_pool.shape}")
    else:
        print("Normality pool already built or CA disabled.")
else:
    print("Contrastive attention disabled — skipping pool build.")

## 7. Optimizer & Scheduler

In [ ]:
from modules.optimizers import build_optimizer, build_lr_scheduler

optimizer  = build_optimizer(args, model)
scheduler  = build_lr_scheduler(args, optimizer)
print("Optimizer:", optimizer)
print("Scheduler:", scheduler)

## 8. Stage 1 — KG Classifier Pretraining (optional)

In [ ]:
import torch.nn.functional as F

def pretrain_kg_classifier(model, dataloader, args, device, num_epochs):
    """Train only the KG classifier head to predict node presence."""
    kg_builder = model.encoder_decoder.kg_builder
    node_list  = model.encoder_decoder.node_list
    node2idx   = model.encoder_decoder.node2idx

    # Only optimise the KG classifier head + visual extractor
    params = [
        {"params": model.visual_extractor.parameters(), "lr": args.lr_ve},
        {"params": model.kg_classifier.parameters(),   "lr": args.lr_ed},
    ]
    pretrain_opt = torch.optim.Adam(params, weight_decay=args.weight_decay)

    model.train()
    history = []
    for epoch in range(1, num_epochs + 1):
        total_loss, n_batches = 0.0, 0
        for _, images, reports_ids, reports_masks in dataloader:
            images      = images.to(device)
            reports_ids = reports_ids.to(device)

            # Decode report IDs back to text to extract KG labels
            reports_text = tokenizer.decode_batch(reports_ids.cpu().numpy())
            kg_labels = model.encoder_decoder.get_kg_labels(reports_text).to(device)

            pretrain_opt.zero_grad()
            logits = model.kg_classifier(images)
            loss   = F.binary_cross_entropy_with_logits(logits, kg_labels)
            loss.backward()
            pretrain_opt.step()

            total_loss += loss.item()
            n_batches  += 1

        avg_loss = total_loss / max(n_batches, 1)
        history.append(avg_loss)
        print(f"  [KG pretrain] Epoch {epoch:3d}/{num_epochs} | Loss: {avg_loss:.4f}")

    return history


if args.kg_pretrain_epochs > 0:
    print(f"=== Stage 1: KG Pretraining ({args.kg_pretrain_epochs} epochs) ===")
    pretrain_history = pretrain_kg_classifier(
        model, train_dataloader, args, device, args.kg_pretrain_epochs
    )
    # Plot pretraining loss
    plt.figure(figsize=(7, 3))
    plt.plot(pretrain_history, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("BCE Loss")
    plt.title("KG Pretraining Loss")
    plt.tight_layout()
    plt.savefig(os.path.join(args.save_dir, "kg_pretrain_loss.png"), dpi=120)
    plt.show()
else:
    print("KG pretraining disabled (kg_pretrain_epochs=0).")

## 9. Stage 2 — Full Training

In [ ]:
from modules.loss import compute_loss
from modules.metrics import compute_scores


def evaluate(model, dataloader, tokenizer, device):
    """Run beam-search on dataloader, return BLEU/METEOR/ROUGE metrics."""
    model.eval()
    gts, res = {}, {}
    with torch.no_grad():
        for step, (ids, images, reports_ids, reports_masks) in enumerate(dataloader):
            images      = images.to(device)
            reports_ids = reports_ids.to(device)
            reports_masks = reports_masks.to(device)

            output, _ = model(images, mode="sample")
            reports   = model.tokenizer.decode_batch(output.cpu().numpy())
            ground_truths = model.tokenizer.decode_batch(reports_ids.cpu().numpy())

            for i, sid in enumerate(ids):
                gts[sid] = [ground_truths[i]]
                res[sid] = [reports[i]]

    return compute_scores(gts, res)


def train_one_epoch(model, dataloader, optimizer, device, kg_loss_weight):
    model.train()
    total_loss = 0.0
    for ids, images, reports_ids, reports_masks in dataloader:
        images        = images.to(device)
        reports_ids   = reports_ids.to(device)
        reports_masks = reports_masks.to(device)

        optimizer.zero_grad()
        output, kg_align_loss = model(images, reports_ids, mode="train")
        ce_loss = compute_loss(output, reports_ids, reports_masks)
        loss = ce_loss + kg_loss_weight * kg_align_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


print("Functions defined.")

In [ ]:
import time

history = {
    "train_loss": [],
    "val_BLEU_4": [],
    "val_METEOR": [],
    "val_ROUGE_L": [],
}

best_val_bleu4 = -1.0
best_epoch     = 0
no_improve     = 0
best_ckpt_path = os.path.join(args.save_dir, "best_model.pth")

print(f"=== Stage 2: Report Generation Training ({args.epochs} epochs) ===")

for epoch in range(1, args.epochs + 1):
    t0 = time.time()

    # ── train ──────────────────────────────────────────────────────────────
    train_loss = train_one_epoch(
        model, train_dataloader, optimizer, device, args.kg_loss_weight
    )
    history["train_loss"].append(train_loss)

    # ── validate ───────────────────────────────────────────────────────────
    val_metrics = evaluate(model, val_dataloader, tokenizer, device)
    history["val_BLEU_4"].append(val_metrics["BLEU_4"])
    history["val_METEOR"].append(val_metrics["METEOR"])
    history["val_ROUGE_L"].append(val_metrics["ROUGE_L"])

    elapsed = time.time() - t0
    print(
        f"Epoch {epoch:3d}/{args.epochs} | "
        f"Loss {train_loss:.4f} | "
        f"Val BLEU-4 {val_metrics['BLEU_4']:.4f} | "
        f"METEOR {val_metrics['METEOR']:.4f} | "
        f"ROUGE-L {val_metrics['ROUGE_L']:.4f} | "
        f"{elapsed:.0f}s"
    )

    # ── checkpoint ─────────────────────────────────────────────────────────
    if val_metrics["BLEU_4"] > best_val_bleu4:
        best_val_bleu4 = val_metrics["BLEU_4"]
        best_epoch     = epoch
        no_improve     = 0
        torch.save(
            {
                "epoch": epoch,
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_metrics": val_metrics,
            },
            best_ckpt_path,
        )
        print(f"  ✓ Saved best checkpoint (BLEU-4={best_val_bleu4:.4f})")
    else:
        no_improve += 1
        if no_improve >= args.early_stop:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {args.early_stop} epochs).")
            break

    # ── periodic checkpoint ────────────────────────────────────────────────
    if epoch % args.save_period == 0:
        torch.save(
            {"epoch": epoch, "state_dict": model.state_dict()},
            os.path.join(args.save_dir, f"checkpoint_ep{epoch}.pth"),
        )

    scheduler.step()

print(f"\nTraining complete. Best BLEU-4={best_val_bleu4:.4f} at epoch {best_epoch}.")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history["val_BLEU_4"],  label="BLEU-4")
axes[1].plot(history["val_METEOR"],  label="METEOR")
axes[1].plot(history["val_ROUGE_L"], label="ROUGE-L")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("Validation Metrics")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(args.save_dir, "training_curves.png"), dpi=120)
plt.show()

## 10. Test Evaluation

In [ ]:
# Load the best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} "
      f"(Val BLEU-4={ckpt['val_metrics']['BLEU_4']:.4f})")

test_metrics = evaluate(model, test_dataloader, tokenizer, device)

print("\n=== Test Results ===")
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.4f}")

In [ ]:
# Save test metrics to JSON
results_path = os.path.join(args.save_dir, "test_results.json")
with open(results_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print(f"Test results saved to {results_path}")

## 11. Qualitative Inspection

In [ ]:
model.eval()

# Grab the first batch from the test set
ids, images, reports_ids, reports_masks = next(iter(test_dataloader))
images      = images.to(device)
reports_ids = reports_ids.to(device)

with torch.no_grad():
    output, _ = model(images, mode="sample")

generated    = model.tokenizer.decode_batch(output.cpu().numpy())
ground_truth = model.tokenizer.decode_batch(reports_ids.cpu().numpy())

n_show = min(4, len(ids))
for i in range(n_show):
    # Show both X-rays side by side
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    for j in range(2):
        img_tensor = images[i, j].cpu().permute(1, 2, 0).numpy()
        # Undo ImageNet normalisation for display only
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img_tensor = np.clip(img_tensor * std + mean, 0, 1)
        axes[j].imshow(img_tensor)
        axes[j].axis("off")
    fig.suptitle(f"Study ID: {ids[i]}", fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"[GT]  {ground_truth[i]}")
    print(f"[GEN] {generated[i]}")
    print("-" * 80)

## 12. Resume from Checkpoint (optional)

In [ ]:
# To resume training from a specific checkpoint, run this cell
# before the training loop (Section 9) and set args.resume.

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    if optimizer and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    print(f"Resumed from {path} (epoch {start_epoch - 1})")
    return start_epoch

# Example usage:
# start_epoch = load_checkpoint("results/iu_xray_kg/best_model.pth", model, optimizer)
print("load_checkpoint helper defined. Uncomment above to use.")